# 🔧 CodeFix Agent
**Autonomous code debugging agent — LangGraph + MCP-style tool loop**

Architecture (mirrors published SWE-agent / RepairAgent research):
```
stack_trace/error
       ↓
  [INVESTIGATE]  — reads file, greps for context
       ↓
  [HYPOTHESIZE]  — LLM proposes root cause + patch
       ↓
  [VALIDATE]     — runs tests, loops back if still failing
       ↓
  [REPORT]       — root cause, diff, confidence, tests passed
```

**Free to run:** uses `claude-haiku-3` (cheapest Anthropic model, ~$0.001 per run).
No vector DB, no external infra — just the Anthropic SDK + LangGraph.

---
### Setup
```bash
pip install anthropic langgraph
```
Set your API key in Cell 2.

In [ ]:
# Cell 1 — Install dependencies
!pip install anthropic langgraph -q

In [ ]:
# Cell 2 — API key (free tier works; haiku is ~$0.001/run)
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-YOUR_KEY_HERE"  # ← paste your key

In [ ]:
# Cell 3 — MCP-style tool definitions (file read, grep, test runner)
import subprocess, re, difflib, textwrap
from pathlib import Path

# ── Tool 1: Read a file ───────────────────────────────────────────────────────
def tool_read_file(path: str) -> str:
    """Read source file from disk with line numbers."""
    p = Path(path)
    if not p.exists():
        return f"ERROR: file not found → {path}"
    lines = p.read_text().splitlines()
    return "\n".join(f"{i+1:4d}  {l}" for i, l in enumerate(lines))

# ── Tool 2: Grep repo ─────────────────────────────────────────────────────────
def tool_grep(pattern: str, directory: str = ".") -> str:
    """Grep for a pattern in Python files."""
    try:
        result = subprocess.run(
            ["grep", "-rn", "--include=*.py", pattern, directory],
            capture_output=True, text=True, timeout=10
        )
        return result.stdout or "(no matches)"
    except Exception as e:
        return f"grep error: {e}"

# ── Tool 3: Apply patch and run tests ────────────────────────────────────────
def tool_apply_and_test(file_path: str, patched_code: str, test_command: str) -> dict:
    """
    Write patched_code to file_path, run test_command, return result.
    Returns dict: {passed: bool, output: str, diff: str}
    """
    p = Path(file_path)
    original = p.read_text() if p.exists() else ""
    
    # Compute diff for the report
    diff = "".join(difflib.unified_diff(
        original.splitlines(keepends=True),
        patched_code.splitlines(keepends=True),
        fromfile=f"{file_path} (original)",
        tofile=f"{file_path} (patched)",
    ))
    
    # Write the patch
    p.write_text(patched_code)
    
    # Run tests
    try:
        result = subprocess.run(
            test_command.split(), capture_output=True, text=True, timeout=30
        )
        passed = result.returncode == 0
        output = result.stdout + result.stderr
    except Exception as e:
        passed = False
        output = str(e)
    
    return {"passed": passed, "output": output[:2000], "diff": diff}

print("✅ Tools defined")

In [ ]:
# Cell 4 — LangGraph state and node definitions
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
import anthropic, json

client = anthropic.Anthropic()
MODEL  = "claude-haiku-4-5"  # cheapest, fast enough for debugging
MAX_ITERATIONS = 3           # max validate→hypothesize loops

# ── Agent state ───────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    error_input:    str          # stack trace / failing test output
    file_path:      str          # file to fix
    test_command:   str          # e.g. "python -m pytest test_myfile.py -v"
    file_content:   str          # current file content (with line numbers)
    grep_results:   str          # context from grep
    root_cause:     str          # LLM hypothesis
    patched_code:   str          # proposed fix
    test_result:    dict         # {passed, output, diff}
    iteration:      int
    final_report:   str

# ── Node 1: INVESTIGATE ───────────────────────────────────────────────────────
def investigate(state: AgentState) -> AgentState:
    print("\n🔍 [INVESTIGATE] Reading file and grepping for context...")
    content = tool_read_file(state["file_path"])
    
    # Extract the most relevant symbol from the error to grep
    error_lines = state["error_input"].splitlines()
    grep_term = next(
        (re.findall(r'[A-Za-z_][A-Za-z0-9_]+', l)[-1]
         for l in error_lines if "Error" in l or "error" in l),
        "def "
    )
    grep_out = tool_grep(grep_term)
    
    print(f"   File: {state['file_path']} ({len(content.splitlines())} lines)")
    print(f"   Grep term: '{grep_term}'")
    return {**state, "file_content": content, "grep_results": grep_out}

# ── Node 2: HYPOTHESIZE ───────────────────────────────────────────────────────
def hypothesize(state: AgentState) -> AgentState:
    print(f"\n🧠 [HYPOTHESIZE] Iteration {state['iteration']+1}/{MAX_ITERATIONS}")
    
    prior_feedback = ""
    if state.get("test_result") and not state["test_result"]["passed"]:
        prior_feedback = f"""
Previous patch FAILED. Test output:
{state['test_result']['output']}

Revise your hypothesis and try a different fix.
"""
    
    prompt = f"""You are an expert Python debugger. Analyze the error and fix the code.

ERROR / FAILING TEST OUTPUT:
{state['error_input']}

SOURCE FILE ({state['file_path']}):
{state['file_content']}

GREP CONTEXT:
{state['grep_results']}
{prior_feedback}

Respond in this EXACT JSON format (no markdown fences):
{{
  "root_cause": "one-sentence explanation of the bug",
  "confidence": "high|medium|low",
  "patched_code": "the COMPLETE corrected file content as a string"
}}"""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    # Strip any accidental markdown fences
    raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()
    
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: extract fields with regex
        parsed = {
            "root_cause": re.search(r'"root_cause":\s*"([^"]+)"', raw, re.S).group(1) if re.search(r'"root_cause"', raw) else "parse error",
            "confidence": "low",
            "patched_code": raw
        }
    
    print(f"   Root cause: {parsed['root_cause']}")
    print(f"   Confidence: {parsed['confidence']}")
    
    return {**state,
            "root_cause": parsed["root_cause"],
            "patched_code": parsed["patched_code"],
            "iteration": state["iteration"] + 1}

# ── Node 3: VALIDATE ──────────────────────────────────────────────────────────
def validate(state: AgentState) -> AgentState:
    print(f"\n🧪 [VALIDATE] Running: {state['test_command']}")
    result = tool_apply_and_test(
        state["file_path"],
        state["patched_code"],
        state["test_command"]
    )
    status = "✅ PASSED" if result["passed"] else "❌ FAILED"
    print(f"   Tests: {status}")
    if not result["passed"]:
        print(f"   Output (last 500 chars): {result['output'][-500:]}")
    return {**state, "test_result": result}

# ── Node 4: REPORT ────────────────────────────────────────────────────────────
def report(state: AgentState) -> AgentState:
    print("\n📋 [REPORT] Generating final report...")
    tr = state.get("test_result", {})
    success = tr.get("passed", False)
    
    report_text = f"""
╔══════════════════════════════════════════════════════════════╗
║                   CODEFIX AGENT — REPORT                     ║
╚══════════════════════════════════════════════════════════════╝

File:         {state['file_path']}
Status:       {'✅ FIXED — all tests pass' if success else '⚠️  PARTIAL — max iterations reached'}
Iterations:   {state['iteration']}

ROOT CAUSE
──────────
{state.get('root_cause', 'N/A')}

PATCH DIFF
──────────
{tr.get('diff', '(no diff generated)')}

TEST OUTPUT
───────────
{tr.get('output', 'N/A')[:1000]}
"""
    print(report_text)
    return {**state, "final_report": report_text}

# ── Routing logic ─────────────────────────────────────────────────────────────
def should_loop(state: AgentState) -> str:
    if state["test_result"]["passed"]:
        return "report"
    if state["iteration"] >= MAX_ITERATIONS:
        return "report"
    return "hypothesize"   # loop back

print("✅ Nodes defined")

In [ ]:
# Cell 5 — Build the LangGraph state machine
builder = StateGraph(AgentState)

builder.add_node("investigate",  investigate)
builder.add_node("hypothesize",  hypothesize)
builder.add_node("validate",     validate)
builder.add_node("report",       report)

builder.set_entry_point("investigate")
builder.add_edge("investigate", "hypothesize")
builder.add_edge("hypothesize", "validate")
builder.add_conditional_edges("validate", should_loop,
                               {"hypothesize": "hypothesize", "report": "report"})
builder.add_edge("report", END)

graph = builder.compile()
print("✅ Graph compiled")
print("   Nodes:", list(graph.nodes))

In [ ]:
# Cell 6 — Demo: create a buggy file + test, then run the agent
from pathlib import Path

# ── Create a buggy Python file ────────────────────────────────────────────────
buggy_code = '''
def calculate_average(numbers):
    """Return the average of a list of numbers."""
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)   # BUG: no guard for empty list → ZeroDivisionError


def find_max(numbers):
    """Return the maximum value."""
    max_val = numbers[0]          # BUG: IndexError on empty list
    for n in numbers:
        if n > max_val:
            max_val = n
    return max_val
'''

# ── Create a test file ────────────────────────────────────────────────────────
test_code = '''
import pytest
from buggy_module import calculate_average, find_max

def test_average_empty():
    assert calculate_average([]) == 0   # should return 0 for empty, not crash

def test_average_normal():
    assert calculate_average([1, 2, 3]) == 2.0

def test_find_max_empty():
    assert find_max([]) is None         # should return None for empty, not crash

def test_find_max_normal():
    assert find_max([3, 1, 4, 1, 5]) == 5
'''

Path("buggy_module.py").write_text(buggy_code)
Path("test_buggy_module.py").write_text(test_code)

# ── Capture the error from an initial test run ────────────────────────────────
import subprocess
initial_result = subprocess.run(
    ["python", "-m", "pytest", "test_buggy_module.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
error_output = initial_result.stdout + initial_result.stderr
print("Initial test run (before fix):")
print(error_output)

In [ ]:
# Cell 7 — Run the CodeFix Agent!
initial_state: AgentState = {
    "error_input":  error_output,
    "file_path":    "buggy_module.py",
    "test_command": "python -m pytest test_buggy_module.py -v --tb=short",
    "file_content": "",
    "grep_results": "",
    "root_cause":   "",
    "patched_code": "",
    "test_result":  {"passed": False, "output": "", "diff": ""},
    "iteration":    0,
    "final_report": "",
}

print("🚀 Starting CodeFix Agent...")
final_state = graph.invoke(initial_state)
print("\n🏁 Agent finished.")

In [ ]:
# Cell 8 — (Optional) Try on YOUR own buggy file
# Replace these three values and re-run Cell 7

# my_state: AgentState = {
#     "error_input":  """PASTE YOUR STACK TRACE OR FAILING TEST OUTPUT HERE""",
#     "file_path":    "your_file.py",
#     "test_command": "python -m pytest your_test_file.py -v",
#     "file_content": "",
#     "grep_results": "",
#     "root_cause":   "",
#     "patched_code": "",
#     "test_result":  {"passed": False, "output": "", "diff": ""},
#     "iteration":    0,
#     "final_report": "",
# }
# final_state = graph.invoke(my_state)

print("Uncomment the block above and fill in your file/error to debug your own code.")